# 03 - Tree-Based Models: Random Forest and Gradient Boosting

**Project:** Predictive Modeling for Drug Discovery via Virtual Screening  
**Student:** Milica Jeftic (ID: 89211255)  
**Date:** January 2026  
**Dataset:** Kaggle - Drug Discovery Virtual Screening Dataset

---

## Goal of This Notebook

This notebook trains and evaluates two tree-based machine learning models for the virtual screening classification task:

1. **Random Forest** - an ensemble of decision trees trained with bootstrap sampling
2. **Gradient Boosting** - a sequential boosting model that learns from previous errors

The goal is to compare these non-linear models with the Logistic Regression baseline from notebook 02 and examine which molecular/protein descriptors are most important for activity prediction.

---

## Expected Outputs

- Validation and test metrics for Random Forest and Gradient Boosting
- Confusion matrix and ROC curve visualizations
- Feature importance plots for both tree-based models
- Saved trained model files in `models/`
- Saved metrics table in `results/metrics/`


## 1. Environment Setup

This section imports the required libraries, configures reproducibility, and defines project paths.

In [ ]:
# ============================
# Environment & Configuration
# ============================

import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# ----------------------------
# Reproducibility & Warnings
# ----------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------
# Pandas display options
# ----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ----------------------------
# Visualization defaults
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
rcParams["figure.figsize"] = (12, 6)
rcParams["font.size"] = 12

%matplotlib inline

# ----------------------------
# Project paths
# ----------------------------
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks directory."""
    current_path = os.path.abspath(start_path)
    for _ in range(3):
        expected_items = [
            os.path.join(current_path, "data"),
            os.path.join(current_path, "notebooks"),
            os.path.join(current_path, "README.md"),
        ]
        if all(os.path.exists(path) for path in expected_items):
            return current_path
        current_path = os.path.dirname(current_path)
    raise FileNotFoundError("Could not locate the project root directory.")

PROJECT_ROOT = find_project_root(os.getcwd())
DATA_PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_PATH = os.path.join(PROJECT_ROOT, "models")
RESULTS_PATH = os.path.join(PROJECT_ROOT, "results")

print("=" * 60)
print("Environment initialized successfully")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print("-" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_PROCESSED_PATH}")
print(f"Models dir   : {MODELS_PATH}")
print("=" * 60)


## 2. Load Preprocessed Data

The processed train, validation, and test splits are loaded from notebook 01. These features have already been scaled using a `StandardScaler` fitted only on the training set.

In [ ]:
print("=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)

X_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_test.csv"))

y_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_train.csv")).squeeze("columns")
y_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_val.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_test.csv")).squeeze("columns")

assert len(X_train) == len(y_train), "Train X/y length mismatch"
assert len(X_val) == len(y_val), "Validation X/y length mismatch"
assert len(X_test) == len(y_test), "Test X/y length mismatch"
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "Feature columns do not match across splits"

feature_cols = list(X_train.columns)

print("Data loaded successfully")
print(f"Training set   : X={X_train.shape}, y={y_train.shape}")
print(f"Validation set : X={X_val.shape}, y={y_val.shape}")
print(f"Test set       : X={X_test.shape}, y={y_test.shape}")
print(f"Number of features: {len(feature_cols)}")

print()
print("Class proportions:")
for split_name, split_y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{split_name}: {split_y.value_counts(normalize=True).sort_index().to_dict()}")


## 3. Helper Functions

The same metric set is used for both models so that validation and test results can be compared directly.

In [ ]:
def evaluate_classifier(model, X, y, model_name, split_name):
    """Evaluate a classifier and return metrics plus prediction arrays."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    metrics = {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1_score": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_proba),
    }
    return metrics, y_pred, y_proba


def plot_confusion_and_roc(y_true, y_pred, y_proba, title, output_path):
    """Save a confusion matrix and ROC curve in one figure."""
    cm = confusion_matrix(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        ax=axes[0],
        cbar=False,
        xticklabels=["Inactive", "Active"],
        yticklabels=["Inactive", "Active"],
    )
    axes[0].set_title(f"Confusion Matrix - {title}", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")

    axes[1].plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {auc:.4f}")
    axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random Classifier")
    axes[1].set_title(f"ROC Curve - {title}", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(loc="lower right")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_feature_importance(model, feature_names, title, output_path, top_n=10):
    """Plot the top feature importances for a fitted tree-based model."""
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance.head(top_n), x="importance", y="feature")
    plt.title(title, fontsize=12, fontweight="bold")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return importance


## 4. Train Tree-Based Models

Two models are trained using the same training data:

- `RandomForestClassifier` uses balanced class weights to account for the class imbalance.
- `GradientBoostingClassifier` does not have a `class_weight` parameter, so balanced sample weights are provided during fitting.

The settings are intentionally moderate so the models remain easy to explain and fast to reproduce.

In [ ]:
print("=" * 60)
print("TRAINING TREE-BASED MODELS")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE,
)

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

training_times = {}

start = time.time()
rf_model.fit(X_train, y_train)
training_times["Random Forest"] = time.time() - start

start = time.time()
gb_model.fit(X_train, y_train, sample_weight=sample_weights)
training_times["Gradient Boosting"] = time.time() - start

print("Training completed")
for model_name, elapsed in training_times.items():
    print(f"{model_name}: {elapsed:.4f} seconds")


## 5. Validation Evaluation

Both models are evaluated on the validation set before looking at the held-out test set. This keeps model comparison separate from final test evaluation.

In [ ]:
print("=" * 60)
print("VALIDATION EVALUATION")
print("=" * 60)

validation_results = []
validation_predictions = {}

for model_name, model in [("Random Forest", rf_model), ("Gradient Boosting", gb_model)]:
    metrics, y_pred, y_proba = evaluate_classifier(model, X_val, y_val, model_name, "validation")
    validation_results.append(metrics)
    validation_predictions[model_name] = {"pred": y_pred, "proba": y_proba}

validation_metrics_df = pd.DataFrame(validation_results)
display(validation_metrics_df)

best_validation_model_name = validation_metrics_df.sort_values(
    ["roc_auc", "f1_score"], ascending=False
).iloc[0]["model"]

print(f"Best validation model by ROC-AUC/F1: {best_validation_model_name}")


## 6. Test Evaluation

The held-out test set is used after validation comparison. Both models are evaluated to provide a direct comparison with the baseline and with each other.

In [ ]:
print("=" * 60)
print("TEST EVALUATION")
print("=" * 60)

test_results = []
test_predictions = {}

for model_name, model in [("Random Forest", rf_model), ("Gradient Boosting", gb_model)]:
    metrics, y_pred, y_proba = evaluate_classifier(model, X_test, y_test, model_name, "test")
    test_results.append(metrics)
    test_predictions[model_name] = {"pred": y_pred, "proba": y_proba}

test_metrics_df = pd.DataFrame(test_results)
display(test_metrics_df)


## 7. Visual Evaluation

Confusion matrices and ROC curves are saved for each model on the held-out test set.

In [ ]:
figures_path = os.path.join(RESULTS_PATH, "figures")
os.makedirs(figures_path, exist_ok=True)

for model_name in ["Random Forest", "Gradient Boosting"]:
    safe_name = model_name.lower().replace(" ", "_")
    output_path = os.path.join(figures_path, f"{safe_name}_test_evaluation.png")
    plot_confusion_and_roc(
        y_test,
        test_predictions[model_name]["pred"],
        test_predictions[model_name]["proba"],
        f"{model_name} Test Set",
        output_path,
    )
    print(f"Saved {model_name} test visualization to: {output_path}")


## 8. Feature Importance

Tree-based models provide feature importance scores. These scores help identify which descriptors contribute most to activity prediction, although they should be interpreted as model-specific importance rather than causal explanations.

In [ ]:
rf_importance_path = os.path.join(figures_path, "random_forest_feature_importance.png")
gb_importance_path = os.path.join(figures_path, "gradient_boosting_feature_importance.png")

rf_importance = plot_feature_importance(
    rf_model,
    feature_cols,
    "Random Forest Feature Importance",
    rf_importance_path,
)

gb_importance = plot_feature_importance(
    gb_model,
    feature_cols,
    "Gradient Boosting Feature Importance",
    gb_importance_path,
)

print("Top Random Forest features:")
display(rf_importance.head(10))

print("Top Gradient Boosting features:")
display(gb_importance.head(10))


## 9. Save Models and Metrics

The trained models and metric tables are saved for later model comparison in notebook 05.

In [ ]:
print("=" * 60)
print("SAVING TREE MODEL OUTPUTS")
print("=" * 60)

os.makedirs(MODELS_PATH, exist_ok=True)
metrics_path = os.path.join(RESULTS_PATH, "metrics")
os.makedirs(metrics_path, exist_ok=True)

rf_model_path = os.path.join(MODELS_PATH, "random_forest.joblib")
gb_model_path = os.path.join(MODELS_PATH, "gradient_boosting.joblib")

joblib.dump(rf_model, rf_model_path)
joblib.dump(gb_model, gb_model_path)

all_tree_metrics = pd.concat([validation_metrics_df, test_metrics_df], ignore_index=True)
tree_metrics_path = os.path.join(metrics_path, "tree_models_metrics.csv")
all_tree_metrics.to_csv(tree_metrics_path, index=False)

rf_importance.to_csv(os.path.join(metrics_path, "random_forest_feature_importance.csv"), index=False)
gb_importance.to_csv(os.path.join(metrics_path, "gradient_boosting_feature_importance.csv"), index=False)

print(f"Random Forest model saved to: {rf_model_path}")
print(f"Gradient Boosting model saved to: {gb_model_path}")
print(f"Tree model metrics saved to: {tree_metrics_path}")
display(all_tree_metrics)


## 10. Tree Model Summary

This notebook trained Random Forest and Gradient Boosting models as non-linear alternatives to the Logistic Regression baseline. Both models use the same train/validation/test split prepared in notebook 01, so their results are directly comparable with the baseline model.

### Key Results

Both tree-based models achieved perfect validation and test performance on the current processed dataset:

- Random Forest validation/test accuracy, F1-score, and ROC-AUC: 1.0000
- Gradient Boosting validation/test accuracy, F1-score, and ROC-AUC: 1.0000

This means that both models correctly classified all validation and test samples, with no false positives and no false negatives.

### Feature Importance

The feature importance analysis shows that `binding_affinity` is the dominant predictor. Random Forest also uses `logp_pi_interaction` and `logp`, while Gradient Boosting relies almost entirely on `binding_affinity`.

This suggests that the target label (`active`) is very strongly related to binding affinity and closely associated descriptors. Because this is a synthetic dataset, the classification problem is highly separable and tree-based models can learn the decision boundary almost perfectly.

### Interpretation and Limitation

The perfect results are not treated as a general expectation for real drug discovery data. In a real virtual screening workflow, binding affinity may not always be available before prediction, and if the activity label was derived from binding affinity, using it as an input feature can make the task artificially easy.

Therefore, the tree-based models perform best on this dataset, but the final report should discuss this as an important dataset limitation. The saved metrics, models, and feature importance files will be reused in notebook 05 for final model comparison and error analysis.
